# باب 7: چیٹ ایپلیکیشنز کی تعمیر
## مائیکروسافٹ فاؤنڈری ماڈلز API فوری آغاز

یہ نوٹ بک [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) سے لیا گیا ہے جس میں وہ نوٹ بکس شامل ہیں جو [Azure OpenAI](notebook-azure-openai.ipynb) خدمات تک رسائی فراہم کرتے ہیں۔

> **نوٹ:** GitHub Models جولائی 2026 کے آخر میں بند ہو رہا ہے۔ یہ نوٹ بک اب [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) کی طرف ہدف بناتی ہے، جو وہی مفت کوشش کے لیے ماڈل کیٹلاگ اور Azure AI Inference SDK تجربہ فراہم کرتا ہے۔


# جائزہ  
"بڑے زبان ماڈلز ایسے فنکشنز ہوتے ہیں جو متن کو متن میں تبدیل کرتے ہیں۔ دی گئی ان پٹ ٹیکسٹ کی سٹرنگ کے حساب سے، ایک بڑا زبان ماڈل اگلے آنے والے متن کی پیش گوئی کرنے کی کوشش کرتا ہے"(1)۔ یہ "کويڪ اسٹارٹ" نوٹ بک صارفین کو اعلیٰ سطح کے LLM تصورات، AML کے ساتھ شروع کرنے کے لیے بنیادی پیکج کی ضروریات، پرامپٹ ڈیزائن کا ایک نرم تعارف، اور مختلف استعمال کے چند مختصر مثالیں متعارف کرائے گی۔  


## فہرست مضامین  

[جائزہ](#overview)  
[اوپن اے آئی سروس کیسے استعمال کریں](#how-to-use-openai-service)  
[1. اپنی اوپن اے آئی سروس بنانا](#1.-creating-your-openai-service)  
[2. تنصیب](#2.-installation)    
[3. اسناد](#3.-credentials)  

[استعمال کے کیسز](#use-cases)    
[1. متن کا خلاصہ بنائیں](#1.-summarize-text)  
[2. متن کی درجہ بندی کریں](#2.-classify-text)  
[3. نئے مصنوعاتی نام بنائیں](#3.-generate-new-product-names)  
[4. ایک درجہ بند کرنے والے کو بہتر بنائیں](#4.fine-tune-a-classifier)  

[حوالہ جات](#references)


### اپنا پہلا پرامپٹ بنائیں  
یہ مختصر مشق Microsoft Foundry Models میں ایک ماڈل کو "خلاصہ" کے ایک سادہ کام کے لئے پرامپٹس بھیجنے کا بنیادی تعارف فراہم کرے گی۔  


**مراحل**:  
1. اگر آپ نے ابھی تک نہیں کیا ہے تو اپنے پائتھون ماحول میں `azure-ai-inference` لائبریری نصب کریں۔  
2. معیاری ہیلپر لائبریریز لوڈ کریں اور اپنے Microsoft Foundry Models کے اسناد مرتب کریں۔  
3. اپنے کام کے لیے ایک ماڈل منتخب کریں  
4. ماڈل کے لئے ایک سادہ پرامپٹ بنائیں  
5. ماڈل API کو اپنی درخواست جمع کرائیں!  


### 1. `azure-ai-inference` انسٹال کریں


In [ ]:
%pip install azure-ai-inference

### 2. مددگار لائبریریاں درآمد کریں اور اسناد کا آغاز کریں


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. درست ماڈل تلاش کرنا  
GPT-4o اور GPT-4o mini جیسے ماڈل قدرتی زبان کو سمجھ سکتے ہیں اور پیدا کر سکتے ہیں، اور یہ مائیکروسافٹ فاؤنڈری ماڈلز کے کیٹلاگ میں میٹا، میسٹرال، کوہیر، اور مائیکروسافٹ کے ماڈلز کے ساتھ دستیاب ہیں۔


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. پرامپٹ ڈیزائن  

"بڑے زبان ماڈلز کا جادو یہ ہے کہ وسیع مقدار میں متن پر پیش گوئی کی اس غلطی کو کم کرنے کی تربیت دے کر، ماڈلز ان پیش گوئیوں کے لیے مفید تصورات سیکھ لیتے ہیں۔ مثال کے طور پر، وہ یہ تصورات سیکھتے ہیں"(1):

* کیسے ہجے کریں  
* گرامر کیسے کام کرتا ہے  
* کیسے پیرا فریز کریں  
* سوالات کا جواب کیسے دیں  
* گفتگو کیسے کریں  
* کئی زبانوں میں کیسے لکھیں  
* کوڈ کیسے کریں  
* وغیرہ  

#### بڑے زبان ماڈل کو کیسے کنٹرول کریں  
"بڑے زبان ماڈل میں تمام ان پٹس میں سب سے زیادہ اثر انداز ہونے والا ان پٹ ٹیکسٹ پرامپٹ ہے(1)۔

بڑے زبان ماڈلز کو چند طریقوں سے آؤٹ پٹ پیدا کرنے کے لیے پرامپٹ کیا جا سکتا ہے:

ہدایت: ماڈل کو بتائیں کہ آپ کیا چاہتے ہیں  
تکمیل: ماڈل کو اس بات کی تکمیل کرانے کی ترغیب دیں جو آپ چاہتے ہیں  
مظاہرہ: ماڈل کو دکھائیں کہ آپ کیا چاہتے ہیں، جس میں شامل ہو:
پرامپٹ میں چند مثالیں  
فائن ٹیوننگ ٹریننگ ڈیٹا سیٹ میں سیکڑوں یا ہزاروں مثالیں"



#### پرامپٹ بنانے کے تین بنیادی اصول ہیں:

**دکھائیں اور بتائیں**۔ واضح کریں کہ آپ کیا چاہتے ہیں چاہے ہدایات کے ذریعے، مثالوں کے ذریعے، یا دونوں کا امتزاج۔ اگر آپ چاہتے ہیں کہ ماڈل آئٹمز کی فہرست کو حروف تہجی کی ترتیب میں مرتب کرے یا پیراگراف کو جذبے کی بنا پر درجہ بندی کرے، تو اسے دکھائیں کہ آپ یہی چاہتے ہیں۔

**معیاری ڈیٹا فراہم کریں**۔ اگر آپ کوئی درجہ بندی کار بنانے کی کوشش کر رہے ہیں یا چاہتے ہیں کہ ماڈل کسی پیٹرن کی پیروی کرے، تو یقینی بنائیں کہ مثالیں کافی ہوں۔ اپنے نمونوں کو ضرور پروف ریڈ کریں — ماڈل عام طور پر بنیادی ہجے کی غلطیوں کو سنبھال لیتا ہے اور جواب دے دیتا ہے، لیکن ممکن ہے کہ یہ سمجھے کہ یہ جان بوجھ کر کیا گیا ہے اور اس کا اثر جواب پر پڑ سکتا ہے۔

**اپنی ترتیبات چیک کریں۔** ٹمپریچر اور top_p کی ترتیبات ماڈل کی جواب پیدا کرنے میں کتنی معین ہے، کو کنٹرول کرتی ہیں۔ اگر آپ ایسے جواب کے لئے کہہ رہے ہیں جس کے لئے صرف ایک درست جواب ہو، تو آپ ان کو کم سیٹ کریں گے۔ اگر آپ زیادہ متنوع جوابات چاہتے ہیں، تو آپ انہیں زیادہ بھی سیٹ کر سکتے ہیں۔ سب سے عام غلطی جو لوگ کرتے ہیں وہ یہ سمجھنا ہے کہ یہ ترتیبات "چالاکی" یا "تخلیقی صلاحیت" کو کنٹرول کرتی ہیں۔


ماخذ: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. جمع کرائیں!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### ایک ہی کال کو دہرائیں، نتائج کا موازنہ کیسے ہوتا ہے؟


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## متن کا خلاصہ  
#### چیلنج  
متن کے اختتام پر 'tl;dr:' شامل کر کے متن کا خلاصہ کریں۔ غور کریں کہ ماڈل بغیر اضافی ہدایات کے کئی کام کیسے انجام دیتا ہے۔ آپ ماڈل کے رویے کو تبدیل کرنے اور موصولہ خلاصہ کو اپنی مرضی کے مطابق بنانے کے لیے tl;dr سے زیادہ وضاحتی اشارے آزما سکتے ہیں(3)۔  

حالیہ کام نے زبان کی بڑی مقدار پر پری ٹریننگ اور پھر مخصوص کام پر فائن ٹوننگ کے ذریعے کئی NLP کاموں اور بنچ مارکس پر قابل ذکر ترقی دکھائی ہے۔ اگرچہ عموماً یہ طریقہ کار معماری کے لحاظ سے کام سے آزاد ہوتا ہے، لیکن پھر بھی اس طریقہ کار کو ہزاروں یا دس ہزاروں مثالوں کے مخصوص کام کے فائن ٹوننگ ڈیٹا سیٹس کی ضرورت ہوتی ہے۔ اس کے برعکس، انسان عام طور پر چند مثالوں یا آسان ہدایات کی مدد سے نیا زبانی کام انجام دے سکتے ہیں - ایسا کچھ جو موجودہ NLP سسٹم ابھی تک بڑی حد تک مشکل محسوس کرتے ہیں۔ یہاں ہم دکھاتے ہیں کہ زبان کے ماڈلز کو بڑھانے سے کام سے آزاد، چند شوٹ کی کارکردگی بہت بہتر ہو جاتی ہے، کبھی کبھار یہ پچھلے بہترین فائن ٹوننگ طریقوں کے ساتھ مسابقت بھی کر لیتا ہے۔  



خلاصہ  


# متعدد استعمال کے کیسز کے لیے مشقیں  
1. متن کا خلاصہ کریں  
2. متن کی درجہ بندی کریں  
3. نئے پروڈکٹ نام تخلیق کریں


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## متن کو درجہ بندی کریں  
#### چیلنج  
آئٹمز کو ایسی اقسام میں درجہ بند کریں جو استنباط کے وقت فراہم کی گئی ہوں۔ مندرجہ ذیل مثال میں، ہم درجہ بند کرنے کے لیے پرامپٹ (*playground_reference) میں دونوں اقسام اور متن فراہم کرتے ہیں۔ 

صارف کی استفسار: ہیلو، میرے لیپ ٹاپ کی بورڈ کی ایک چابی حال ہی میں ٹوٹ گئی ہے اور مجھے اس کی جگہ ضرورت ہوگی:  

درجہ بند کردہ قسم:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## نئے پروڈکٹ نام تیار کریں
#### چیلنج
مثالوں کے الفاظ سے پروڈکٹ کے نام تیار کریں۔ یہاں ہم پرامپٹ میں اس پروڈکٹ کے بارے میں معلومات شامل کرتے ہیں جس کے لیے ہم نام تیار کرنے جا رہے ہیں۔ ہم ایک مشابہہ مثال بھی فراہم کرتے ہیں تاکہ وہ پیٹرن دکھایا جا سکے جو ہم حاصل کرنا چاہتے ہیں۔ ہم نے رینڈم نیس بڑھانے اور مزید تخلیقی جوابات حاصل کرنے کے لیے ٹیمپریچر ویلیو بھی زیادہ رکھی ہے۔

پروڈکٹ کی تفصیل: گھریلو ملک شیک بنانے والا
بیج کے الفاظ: تیز، صحت مند، کمپیکٹ۔
پروڈکٹ نام: HomeShaker, Fit Shaker, QuickShake, Shake Maker

پروڈکٹ کی تفصیل: جوتوں کا ایک جوڑا جو کسی بھی قدم کے سائز پر فٹ ہو سکتا ہے۔
بیج کے الفاظ: قابل مطابقت، فٹ، اومن-فٹ۔


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# حوالہ جات  
- [اوپن اے آئی کوک بک](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [مائیکروسافٹ فاؤنڈری پورٹل](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [متن کی درجہ بندی کے لیے GPT ماڈلز کو بہتر بنانے کی بہترین عملی حکمتِ عملیاں](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# مزید مدد کے لیے  
[اوپن اے آئی کمرشلائزیشن ٹیم](AzureOpenAITeam@microsoft.com) 


# شراکت دار
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
